# HashiCorp Vault -- exercices

## Set Up

### 1. Ajout de la clé GPG d'HashiCorp

Cette étape garantit que le logiciel que tu vas télécharger provient bien de l'éditeur et n'a pas été altéré.

```Bash
wget -O- https://apt.releases.hashicorp.com/gpg | sudo gpg --dearmor -o /usr/share/keyrings/hashicorp-archive-keyring.gpg
```

### 2. Ajout du dépôt officiel

Nous déclarons le dépôt d'HashiCorp dans les sources de ton gestionnaire de paquets.

```Bash
echo "deb [signed-by=/usr/share/keyrings/hashicorp-archive-keyring.gpg] https://apt.releases.hashicorp.com $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/hashicorp.list
```

### 3. Installation de Vault

Mise à jour de l'index et installation du binaire natif.

```Bash
sudo apt update && sudo apt install vault
```

### 4. Démarrage de l'infrastructure

Dans ton terminal système, lance la commande suivante et garde le Root Token affiché sous la main.

```Bash
vault server -dev
```

Ne ferme pas ce Terminal.
Installe les dépendances dans un autre terminal.

```Bash
uv add hvac requests
```

### 5. Variable d’environnement

L'architecture de configuration de Vault Agent accepte l'injection de paramètres via des fichiers, des drapeaux CLI, et des variables d'environnement par ordre de priorité croissante. Les scripts Python liront ces variables d'environnement (`VAULT_ADDR`, `VAULT_TOKEN`). Le client HVAC peut par exemple interroger le backend système pour lire le statut de leader de l'infrastructure afin de valider la connectivité :


In [ ]:
import os
import hvac

# 1. Définition éphémère pour le serveur de développement local
# Remplacer la valeur par le jeton "Root Token" fourni par le terminal Vault
os.environ["VAULT_ADDR"] = "http://127.0.0.1:8200"
os.environ["VAULT_TOKEN"] = (
    "hvs.MON_TOKEN_EPHEMERE_LOCAL"  # À remplacer avec le vrai token éphémére
)

# 2. Initialisation du client administratif HVAC
# Le client lit de manière standard les variables d'environnement
client = hvac.Client(url=os.getenv("VAULT_ADDR"), token=os.getenv("VAULT_TOKEN"))

# 3. Validation du statut du serveur
leader_status = client.sys.read_leader_status()
print(
    f"Connexion réussie. Le serveur est prêt (Leader HA: {leader_status['ha_enabled']})"
)

Connexion réussie. Le serveur est prêt (Leader HA: False)


### ⚠️ Note de Sécurité et d'Architecture : Gestion des Variables

Le document de référence précise explicitement que déployer des jetons Vault en dur dans le code d'une application est une faille de sécurité majeure. Dans une architecture de production réelle, le code Python doit être totalement agnostique : il se contente de lire les variables d'environnement (`VAULT_ADDR`, `VAULT_TOKEN`) préalablement injectées par l'infrastructure (via un fichier `.env`, l'orchestrateur de conteneurs, ou le démon Vault Agent).

**Choix pédagogique :** Puisque ce tutoriel s'appuie sur un serveur Vault local en mode développement (`-dev`), le jeton racine est volatile et change à chaque redémarrage. Pour fluidifier l'exécution de ces exercices, nous allons artificiellement injecter ce jeton dans l'environnement de la session Jupyter courante via le module `os` de Python. Ne reproduisez jamais ce schéma d'injection directe dans vos dépôts d'entreprise.


## Niveau 1 : Fondamentaux du Moteur Key/Value (KV-V2)

### 1. Activation, Configuration et Mécanisme CAS (Check-And-Set)

La première étape consiste à configurer le moteur de secrets. L'API sys/mounts/ permet de créer un point de montage et d'y attacher le plugin kv en spécifiant la version 2. Via HVAC, il est possible d'activer le moteur et d'appliquer immédiatement des règles de configuration strictes pour l'ensemble du chemin.


In [ ]:
import hvac

# 1. Activation sécurisée (Idempotente)
try:
    client.sys.enable_secrets_engine(
        backend_type="kv", path="shared", options={"version": "2"}
    )
    print("Moteur KV-v2 activé sur le chemin 'shared/'.")
except hvac.exceptions.InvalidRequest as e:
    if "path is already in use" in str(e):
        print("Le moteur est déjà activé. Poursuite de la configuration...")
    else:
        raise

# 2. Configuration stricte de l'historique et du Check-And-Set (CAS)
client.secrets.kv.v2.configure(max_versions=20, cas_required=True, mount_point="shared")
print("Configuration de l'historique et du CAS (Check-And-Set) appliquée avec succès.")

Moteur KV-v2 activé sur le chemin 'shared/'.
Configuration de l'historique et du CAS (Check-And-Set) appliquée avec succès.


Le paramètre `cas_required=True` est crucial pour éviter la perte de données concurrentes. Lorsqu'il est actif, le client doit prouver qu'il connaît la version actuelle du secret avant de la modifier. Si un développeur tente de mettre à jour un secret alors que la valeur du paramètre cas fournie ne correspond pas à la version courante sur le serveur, l'API lèvera une exception `hvac.exceptions.InvalidRequest`. Si un secret est créé pour la première fois, le paramètre cas doit être fixé à 0.

### 2. Écriture, Extraction et Métadonnées

L'architecture sépare conceptuellement les données du secret de ses métadonnées et de sa structure. Les développeurs peuvent interroger l'arborescence sans exposer les données via l'endpoint `/subkeys/`. Lors de l'appel à cet endpoint, Vault récupère les secrets, mais remplace la valeur sous-jacente des clés feuille (non-map) par une valeur `null` (ou `nil` selon le client).


In [ ]:
# Création initiale sécurisée d'un secret (cas=0 car le secret n'existe pas encore)
client.secrets.kv.v2.create_or_update_secret(
    mount_point="shared",
    path="dev/square-api",
    secret={"prod": "5678", "sandbox": "1234"},
    cas=0,
)
print("Secret 'dev/square-api' créé avec succès.")

# Lecture complète de la version la plus récente
secret_resp = client.secrets.kv.v2.read_secret_version(
    mount_point="shared", path="dev/square-api"
)

# Extraction des informations pertinentes
print(f"Clés disponibles dans le secret : {secret_resp['data']['data'].keys()}")
print(f"Date de création : {secret_resp['data']['metadata']['created_time']}")
print(f"Version actuelle du secret : {secret_resp['data']['metadata']['version']}")

Secret 'dev/square-api' créé avec succès.
Clés disponibles dans le secret : dict_keys(['prod', 'sandbox'])
Date de création : 2026-08-13T07:58:11.549180403Z
Version actuelle du secret : 1


/tmp/ipykernel_90974/2981975496.py:11: DeprecationWarning: The raise_on_deleted_version parameter will change its default value to False in hvac v3.0.0. The current default of True will preserve previous behavior. To use the old behavior with no warning, explicitly set this value to True. See https://github.com/hvac/hvac/pull/907
  secret_resp = client.secrets.kv.v2.read_secret_version(


Une fois que tu as vérifié que la version actuelle retournée est bien la version 1.

Si un second développeur de ton équipe tente de modifier le mot de passe prod de ce même secret en exécutant exactement le même script (avec cas=0)
cela causera un echec.

**Vault est l'unique source de vérité et centralise tout l'état de l'infrastructure**. Si la requête échoue, c'est grâce au mécanisme **Check-And-Set (CAS)** que nous avons imposé côté serveur via le paramètre `cas_required=True`.

Voici la mécanique exacte :

- En envoyant cas=0, le script dit formellement à Vault : "Crée ce secret uniquement s'il n'a jamais existé".
- Puisque ton secret existe maintenant et se trouve à la version 1, Vault va bloquer la transaction et lever une exception `hvac.exceptions.InvalidRequest`.
- Pour que le second développeur puisse modifier le mot de passe prod, il devra impérativement lire le secret d'abord, constater qu'il est en version 1, puis envoyer sa mise à jour avec cas=1.C'est ce qui empêche deux systèmes ou développeurs d'écraser leurs modifications respectives de manière concurrente.

I​l​ ​est​ ​également​ ​possible​ ​d'implémenter​ ​une​ ​politique​ ​centralisée​ ​de​ ​génération​ ​de​ ​mots​ ​de​ ​passe​ ​pour​ ​que​ ​les​ développeurs​ ​obtiennent​ ​des​ ​chaînes​ ​aléatoires​ ​conformes.​ ​La​ ​politique​ ​HCL​ ​est​ ​envoyée​ ​via​ ​une​ ​requête​ ​POST​ ​au​ ​chemin​ `​sys/policies/password/:nom​​`.​ ​Les​ ​règles​ ​peuvent​ ​imposer​​ la ​longueur,​​et ​​un ​​minimum​ ​de ​​caractères ​​minuscules, ​​majuscules ​de ​​chiffres​ ​et ​​de ​​symboles ​​spéciaux​ ​(via​ ​la déclaration​​ `rule "charset" { min-chars = X }`​​). L'endpoint ne retourne aucune donnée en cas de succès.​


### 3. Exploration et Sous-clés (Subkeys)

L'architecture de Vault sépare conceptuellement les données du secret de ses métadonnées.  
Voici comment interroger l'arborescence des secrets sans jamais exposer les mots de passe en clair sur le réseau en utilisant l'endpoint `/subkeys/`. Cette fois-ci, nous n'utilisons pas `hvac`, mais une requête HTTP directe avec la bibliothèque.


In [ ]:
import os
import requests

# L'API remplace la valeur sous-jacente des clés feuille par 'null' pour les masquer
subkeys_endpoint = f"{os.getenv('VAULT_ADDR')}/v1/shared/subkeys/dev/square-api"

subkeys_resp = requests.get(
    subkeys_endpoint, headers={"X-Vault-Token": os.getenv("VAULT_TOKEN")}
)

print("Exploration de la structure (sans les valeurs) :")
print(subkeys_resp.json()["data"]["subkeys"])

Exploration de la structure (sans les valeurs) :
{'prod': None, 'sandbox': None}


l'objectif est de connaître la structure. Mais poussons l'analyse de cette mécanique un peu plus loin.

Pourquoi concevoir un endpoint spécifique au lieu de simplement demander aux développeurs de lire le secret et d'ignorer les valeurs ? La réponse se trouve dans les journaux d'audit (audit logs).

Si un script de CI/CD ou une interface graphique lit le secret complet juste pour vérifier que la clé `prod` existe, Vault enregistre un accès aux données sensibles. Si ce script tourne toutes les 5 minutes, vos logs de sécurité seront saturés de faux positifs d'accès. En utilisant l'endpoint `/subkeys/`, le système prouve que la clé existe tout en remplaçant la valeur par null, ce qui permet de valider la conformité d'une structure de données sans déclencher d'alerte de sécurité pour lecture de mot de passe.

### 4. Politique Centralisée de Mots de Passe

Pour clôturer le dernier mécanisme de contrôle : la génération de mots de passe dictée par le serveur.  
Plutôt que de laisser chaque microservice générer ses propres chaînes aléatoires (avec le risque d'utiliser de mauvais algorithmes), on peut configurer une politique HCL stricte via l'API `sys/policies/password/:nom`.


In [ ]:
import os
import requests

# Définition de la politique HCL (HashiCorp Configuration Language)
password_policy = """
length = 20
rule "charset" {
  charset = "abcdefghijklmnopqrstuvwxyz"
  min-chars = 2
}
rule "charset" {
  charset = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
  min-chars = 2
}
rule "charset" {
  charset = "0123456789"
  min-chars = 2
}
rule "charset" {
  charset = "!@#$%^&*"
  min-chars = 2
}
"""

policy_endpoint = f"{os.getenv('VAULT_ADDR')}/v1/sys/policies/password/norme-entreprise"

# L'endpoint ne retourne aucune donnée (204 No Content) en cas de succès
response = requests.post(
    policy_endpoint,
    headers={"X-Vault-Token": os.getenv("VAULT_TOKEN")},
    json={"policy": password_policy},
)

if response.status_code == 204:
    print("Politique de mot de passe 'norme-entreprise' configurée avec succès.")
else:
    print(f"Échec de la configuration : {response.status_code} - {response.text}")

Politique de mot de passe 'norme-entreprise' configurée avec succès.


## ​Niveau 2 : Intermédiaire - Automatisation M2M avec AppRole​

​Déployer​ ​des​ ​jetons​ ​Vault​ ​en​ ​dur​ ​dans​ ​le​ ​code​ ​d'une​ ​application​ ​est​ ​une​ ​faille​ ​de​ ​sécurité​ majeure.​ ​AppRole​ ​est​ ​la​ ​méthode​ ​d'authentification​ ​privilégiée​ ​pour​ ​les​ ​architectures​ ​"Machine-to-Machine".​​ Elle​ ​repose ​​sur​ ​deux​ ​composantes​​: ​​le​ `​RoleID`​ ​(comparable​ ​à​ ​un​​ login​ public) ​​et​​le​ `​SecretID` (mot de passe éphémère à usage unique).​

​Le​ ​flux​ ​sécurisé​ ​exige​ ​l'utilisation​ ​d'un​ ​mécanisme​ ​de​ ​"Pull",​ ​où​ ​le​ ​système​ ​de​ ​déploiement​ ​pousse​ ​uniquement​ ​le​ ​RoleID​ ​dans​ ​la​ ​machine​ ​cible,​ ​tandis​ ​que​ ​l'application​ ​"tire"​ ​le​ ​SecretID​ ​d'une​ ​source​ ​sécurisée​ ​au​ ​démarrage.​ ​Le​ ​mode​ ​"Push"​ ​(où​ ​un​ ​tiers​ ​génère​ ​le​ ​jeton​ ​final​ ​et​ ​le​ ​pousse​ ​à​ ​l'application)​ ​est​ ​déconseillé​ ​car​ ​il​ ​oblige​ ​ce​ ​système​ ​tiers​ ​à​ ​manipuler​ ​directement​ ​les​ ​informations​ ​d'identification finales.​

### 1. Configuration du Rôle Applicatif


In [8]:
import hvac

# 1. Activation de la méthode d'authentification AppRole (avec idempotence)
try:
    client.sys.enable_auth_method(method_type="approle", path="approle")
    print("Méthode AppRole activée.")
except hvac.exceptions.InvalidRequest as e:
    if "path is already in use" in str(e):
        print("AppRole est déjà activé.")
    else:
        raise

# 2. Création du rôle applicatif avec les contraintes du document
client.auth.approle.create_or_update_approle(
    role_name="backend-microservice",
    bind_secret_id=True,  # Force la présentation du SecretID lors du login
    secret_id_num_uses=1,  # Le SecretID ne peut être utilisé qu'une seule fois
    secret_id_ttl="10m",  # Le SecretID périme 10 minutes après sa création
    secret_id_bound_cidrs=[
        "127.0.0.1/32"
    ],  # Restreint la demande de jeton au réseau interne
    token_policies=["default"],
)
print("Rôle 'backend-microservice' configuré avec ses contraintes de sécurité.")

AppRole est déjà activé.
Rôle 'backend-microservice' configuré avec ses contraintes de sécurité.


En ingénierie de la sécurité, cela répond au **Principe de Moindre Privilège**. Si ton application est compromise (par exemple, via une faille d'injection qui permet à un attaquant d'exécuter du code), l'attaquant hérite des droits de l'application.

Si ton application possédait la permission de lire son propre RoleID ou de générer de nouveaux SecretID, l'attaquant pourrait s'en servir pour forger ses propres accès persistants, totalement indépendants du cycle de vie de ton application. En forçant l'application à ne connaître que l'endpoint de connexion (`/auth/approle/login`), tu garantis qu'une compromission applicative ne se transforme pas en compromission de l'infrastructure d'authentification.


### 2. Authentification Autonome (AppRole)

Maintenant que le rôle est correctement configuré pour accepter les requêtes de ta machine, nous allons simuler le flux d'authentification complet.

Cette étape se divise en deux phases distinctes telles que décrites dans le document :

- Le rôle de l'Administrateur (ou CI/CD) : Il extrait le RoleID (qui ne change pas) et génère un SecretID éphémère (qui périme après 10 minutes et 1 utilisation). Il peut d'ailleurs y attacher des métadonnées pour faciliter l'audit opérationnel.
- Le rôle de l'Application : Elle utilise un nouveau client hvac totalement vierge (sans le jeton d'administration) et s'authentifie de manière autonome en présentant ce couple d'identifiants.


In [ ]:
import os
import hvac

# --- PARTIE 1 : SIMULATION DE L'ADMINISTRATEUR ---

# Extraction du RoleID (identifiant public)
role_id_resp = client.auth.approle.read_role_id(role_name="backend-microservice")
role_id = role_id_resp["data"]["role_id"]

# Génération dynamique du SecretID (Correction : utilisation d'un dictionnaire Python)
secret_id_resp = client.auth.approle.generate_secret_id(
    role_name="backend-microservice",
    metadata={"env": "production", "region": "eu-west"},
)
secret_id = secret_id_resp["data"]["secret_id"]

print(f"RoleID extrait : {role_id[:8]}...")
print(f"SecretID généré (usage unique) : {secret_id[:8]}...")


# --- PARTIE 2 : SIMULATION DE L'APPLICATION ---

# L'application initialise un nouveau client sans token initial
app_client = hvac.Client(url=os.getenv("VAULT_ADDR"))

# Authentification autonome via AppRole
app_client.auth.approle.login(role_id=role_id, secret_id=secret_id)

print(f"Application authentifiée de manière autonome : {app_client.is_authenticated()}")

RoleID extrait : d9ee4341...
SecretID généré (usage unique) : c507dc37...
Application authentifiée de manière autonome : True


## Niveau 3 : Avancé - Cryptographie (Transit) et Response Wrapping

Les développeurs se heurtent fréquemment à la complexité de l'implémentation de solutions cryptographiques. Le moteur de secrets Transit, ou "Encryption as a Service", permet de gérer cette complexité. L'une des exigences strictes de ce moteur est que toutes les données en clair transitant par son API doivent être encodées en base64 pour permettre de sécuriser la transmission de fichiers binaires ou d'images sans corrompre les structures de données JSON de Vault.

La création de clés via l'endpoint `/transit/keys/:name` permet de spécifier le type cryptographique, tel que `aes256-gcm96` (la valeur par défaut) ou `chacha20-poly1305`. Pour les applications, il est vital de configurer les clés selon les besoins de réplication ou de sauvegarde via les drapeaux exportable (qui permet d'exporter toutes les clés valides du trousseau) et `allow_plaintext_backup`. Des paramètres comme `auto_rotate_period` permet d'instaurer des rotations de clés automatisées avec un minimum d'une heure.

> Ici, Vault agit comme un module de sécurité matériel (HSM) logiciel. L'application lui envoie une donnée, Vault la chiffre avec une clé qu'il garde jalousement en mémoire, et renvoie le texte chiffré (ciphertext). L'application n'a jamais accès à la clé de chiffrement elle-même.

Avant de faire notre première opération de chiffrement, nous devons préparer le terrain.


### 1. Activation du Moteur Transit et Création de la Clé

Il active le moteur transit et demande à Vault de générer une clé nommée `payment-key` (qui utilisera par défaut l'algorithme `AES-256-GCM`).


In [ ]:
try:
    # 1. Activation du moteur Transit
    client.sys.enable_secrets_engine(backend_type="transit", path="transit")
    print("Moteur Transit activé.")
except hvac.exceptions.InvalidRequest as e:
    if "path is already in use" in str(e):
        print("Le moteur Transit est déjà activé.")
    else:
        raise

# 2. Création d'une clé de chiffrement nommée (Correction : key_type)
client.secrets.transit.create_key(
    name="payment-key",
    key_type="aes256-gcm96",  # Algorithme standard et robuste
)
print(
    "Clé de chiffrement 'payment-key' générée et stockée en toute sécurité dans Vault."
)

Le moteur Transit est déjà activé.
Clé de chiffrement 'payment-key' générée et stockée en toute sécurité dans Vault.


### 2. Chiffrement (Encryption)

Il faut différencier l'encodage du chiffrement : le Base64 n'offre absolument aucune sécurité (n'importe qui peut le décoder instantanément).

C'est une pure contrainte technique liée au protocole JSON.

L'API de Vault est conçue pour chiffrer n'importe quel type de donnée, pas seulement du texte. Imagine que tu veuilles chiffrer un fichier PDF, une image, une archive `.zip` ou une sauvegarde de base de données. Ces fichiers sont constitués de données binaires brutes.

Si tu essaies d'insérer des données binaires directement dans un objet JSON, le formatage va se briser instantanément (à cause des retours à la ligne invisibles, des guillemets non échappés ou des caractères non imprimables). La requête HTTP plantera avant même d'arriver au moteur de Vault.

L'encodage Base64 prend n'importe quelle donnée (binaire ou texte complexe) et la transforme en une chaîne de caractères totalement inoffensive, composée uniquement de lettres standards (A-Z, a-z), de chiffres (0-9) et des symboles `+` et `/`. Ainsi, le JSON reste parfaitement valide pendant le transfert réseau, peu importe ce que tu demandes à Vault de chiffrer.

Passons à la pratique. Voici le code pour chiffrer ta première donnée en respectant cette contrainte d'encodage :


In [ ]:
import base64

# La donnée sensible que l'on veut chiffrer (ex: un numéro de carte bancaire)
plaintext = "4000-1234-5678-9010"

# 1. Encodage strict en Base64 exigé par Vault
# - plaintext.encode('utf-8') convertit le texte en octets (bytes)
# - base64.b64encode() effectue la transformation mathématique
# - .decode('utf-8') reconvertit le résultat en chaîne de caractères pour le JSON
encoded_text = base64.b64encode(plaintext.encode("utf-8")).decode("utf-8")

# 2. Demande de chiffrement au moteur Transit
encrypt_resp = client.secrets.transit.encrypt_data(
    name="payment-key", plaintext=encoded_text
)

# Extraction du résultat
ciphertext = encrypt_resp["data"]["ciphertext"]

print(f"1. Donnée originale : {plaintext}")
print(f"2. Format Base64 envoyé : {encoded_text}")
print(f"3. Texte chiffré par Vault : {ciphertext}")

1. Donnée originale : 4000-1234-5678-9010
2. Format Base64 envoyé : NDAwMC0xMjM0LTU2NzgtOTAxMA==
3. Texte chiffré par Vault : vault:v1:aJykud3ryHnN5Ck/wKz5En9fmtiAA0juokY4TGcMUO4p/tRPHoZ+y2myD9ToWAM=


### 3. Déchiffrement (Decryption)

Puisque Vault nous oblige à lui parler en Base64 pour le chiffrement, il va logiquement nous répondre avec ce même standard lors du déchiffrement. C'est donc à ton application (ton script Python) de faire l'opération inverse à la réception.

Il va prendre la variable `ciphertext` générée dans la cellule précédente et demander à Vault de la déchiffrer en utilisant la même `payment-key`.


In [ ]:
# 1. Demande de déchiffrement au moteur Transit
decrypt_resp = client.secrets.transit.decrypt_data(
    name="payment-key", ciphertext=ciphertext
)

# 2. Vault nous renvoie la donnée, mais elle est toujours encodée en Base64
decrypted_base64 = decrypt_resp["data"]["plaintext"]

# 3. Décodage final en local pour retrouver le texte brut
# - base64.b64decode() inverse l'encodage
# - .decode('utf-8') convertit les octets en chaîne de caractères
decrypted_text = base64.b64decode(decrypted_base64).decode("utf-8")

print(f"1. Texte chiffré envoyé : {ciphertext}")
print(f"2. Réponse brute de Vault (Base64) : {decrypted_base64}")
print(f"3. Donnée finale récupérée en clair : {decrypted_text}")

1. Texte chiffré envoyé : vault:v1:aJykud3ryHnN5Ck/wKz5En9fmtiAA0juokY4TGcMUO4p/tRPHoZ+y2myD9ToWAM=
2. Réponse brute de Vault (Base64) : NDAwMC0xMjM0LTU2NzgtOTAxMA==
3. Donnée finale récupérée en clair : 4000-1234-5678-9010


Si tout s'est bien passé, ton terminal devrait t'afficher ton numéro de carte bancaire d'origine `(4000-1234-5678-9010)`.


Le dernier concept fondamental à aborder pour le **Niveau 3** : le Response Wrapping (l'enveloppement de réponse).

**Imagine ce scénario :** tu dois transmettre un mot de passe d'accès très critique à un administrateur système humain, ou à un script de déploiement, mais tu ne veux pas l'envoyer en clair par email, ni sur Slack, ni le laisser traîner dans un fichier de logs.

Vault propose d'encapsuler (wrapper) ce secret dans un "jeton à usage unique" (Cubbyhole). Ce jeton a deux propriétés magiques :

Il a une durée de vie très courte (ex: 60 secondes).

Il s'autodétruit dès sa première lecture.

Avant d'écrire la cellule qui implémente cette mécanique, je te soumets ce cas de figure de sécurité : tu envoies ce jeton d'enveloppement à usage unique à ton administrateur système sur Slack. L'administrateur clique sur le lien pour lire le secret, mais Vault lui retourne une erreur `403 Forbidden` ou `invalid token`.

_Sachant que le jeton était valide pour 60 secondes et qu'il a cliqué au bout de 10 secondes, que dois-tu déduire de cet échec technique, et pourquoi cette architecture est-elle le meilleur système d'alerte possible ?_


### 4. Le Response Wrapping (Enveloppement)

Nous allons créer une donnée arbitraire, demander à Vault de l'envelopper dans un jeton jetable avec une durée de vie (TTL) de 60 secondes, puis nous allons le déballer ("unwrap").


In [16]:
import os
import hvac

# 1. Enveloppement de la donnée (L'action de l'émetteur)
# On donne un TTL (Time To Live) très court de 60 secondes
wrap_resp = client.sys.wrap(
    payload={"mot_de_passe_admin": "SuperSecret123!"}, ttl="60s"
)

# Extraction du jeton jetable généré par Vault
wrapping_token = wrap_resp["wrap_info"]["token"]
print(
    f"1. Le secret a été enveloppé. Voici le jeton jetable à transmettre : {wrapping_token[:12]}..."
)

# 2. Déballage de la donnée (L'action du destinataire)
# Le destinataire utilise le jeton jetable pour s'authentifier temporairement
unwrap_client = hvac.Client(url=os.getenv("VAULT_ADDR"), token=wrapping_token)
unwrap_resp = unwrap_client.sys.unwrap()

print(f"2. Secret déballé avec succès : {unwrap_resp['data']}")

1. Le secret a été enveloppé. Voici le jeton jetable à transmettre : hvs.CAESIMtz...
2. Secret déballé avec succès : {'mot_de_passe_admin': 'SuperSecret123!'}


#### **Le Test Ultime**

Pour valider que le token n'est urilisable qu'une seule fois, je te propose un exercice pratique immédiat : réexécute la commande pour l'authentification.


In [17]:
# 3. Tentative de déballage par un "attaquant" (ou le destinataire légitime retardataire)
unwrap_client.sys.unwrap()

InvalidRequest: wrapping token is not valid or does not exist, on post http://127.0.0.1:8200/v1/sys/wrapping/unwrap

Cette erreur est la preuve mathématique que ton architecture de sécurité fonctionne.

Dès que la première instruction `unwrap_client.sys.unwrap()` a été exécutée, Vault a lu la donnée, te l'a renvoyée, et a immédiatement détruit le jeton `wrapping_token`. Ta deuxième tentative (la simulation de l'attaquant) s'est heurtée à un jeton qui n'existe tout simplement plus dans la base de données de Vault. L'API te rejette violemment.


## Niveau 4 : Moteur PKI et Secrets Dynamiques

### 1. L'Administrateur - Configuration du moteur PKI

Cette cellule simule le travail de l'équipe sécurité. Elle active le moteur, crée une autorité de certification racine (CA) valable 10 ans, et configure un rôle qui autorise la génération de certificats uniquement pour le domaine `*.mon-app.local`, avec une durée de vie maximale très courte (1 heure).


In [ ]:
import hvac

try:
    # 1. Activation du moteur PKI
    client.sys.enable_secrets_engine(backend_type="pki", path="pki")
    print("Moteur PKI activé.")

    # Configuration du TTL maximum du moteur (ex: 10 ans pour le CA racine)
    client.sys.tune_mount_configuration(path="pki", max_lease_ttl="87600h")
except hvac.exceptions.InvalidRequest as e:
    if "path is already in use" in str(e):
        print("Le moteur PKI est déjà activé.")
    else:
        raise

# 2. Génération de l'Autorité de Certification Interne (Root CA)
try:
    client.secrets.pki.generate_root(
        type="internal", common_name="app-internal-ca", extra_params={"ttl": "87600h"}
    )
    print("Autorité de Certification (CA) racine générée.")
except Exception:
    print("Le CA racine existe déjà, on continue.")

# 3. Création du Rôle restrictif pour les applications (Correction ici)
client.secrets.pki.create_or_update_role(
    name="role-serveur-web",
    extra_params={
        "allowed_domains": "mon-app.local",  # Le domaine autorisé
        "allow_subdomains": True,
        "max_ttl": "1h",  # Le certificat dynamique ne vivra qu'une heure maximum
        "key_type": "rsa",
        "key_bits": 2048,
    },
)
print("Rôle 'role-serveur-web' configuré avec succès via extra_params.")

Le moteur PKI est déjà activé.
Autorité de Certification (CA) racine générée.
Rôle 'role-serveur-web' configuré avec succès via extra_params.


### 2. L'Application - Demande du Secret Dynamique

Ici, c'est ton script (ou ton serveur web) qui démarre. Il ne possède aucun certificat TLS (HTTPS) sur le disque. Il va donc demander à Vault d'en forger un sur-le-champ.


In [ ]:
# L'application demande un certificat valide pour son domaine spécifique
cert_resp = client.secrets.pki.generate_certificate(
    name="role-serveur-web",
    common_name="api.mon-app.local",
    extra_params={"ttl": "5m"},  # On demande un bail ultra-court de 5 minutes
)

donnees_certificat = cert_resp["data"]
print("--- Secret Dynamique Généré avec Succès ---")
print(f"Durée de vie (Lease) accordée : {cert_resp['lease_duration']} secondes")
print(
    f"\nCertificat Public (Extrait) :\n{donnees_certificat['certificate'][:150]}...\n"
)

--- Secret Dynamique Généré avec Succès ---
Durée de vie (Lease) accordée : 0 secondes

Certificat Public (Extrait) :
-----BEGIN CERTIFICATE-----
MIIDVTCCAj2gAwIBAgIUXMNZhe6LUGDbVQva0fbqk9VTOjYwDQYJKoZIhvcNAQEL
BQAwGjEYMBYGA1UEAxMPYXBwLWludGVybmFsLWNhMB4XDTI2MDgxMzExN...



Si tu exécutes cette Cellule 13 plusieurs fois, tu verras que Vault génère une clé privée et un certificat totalement différents et uniques à chaque appel, sans rien stocker de manière permanente.

_Si le certificat web de l'application expire au bout de 5 minutes et que le code Python de l'application ne contient aucune logique complexe pour surveiller l'horloge système et rappeler l'API de Vault à la minute 4... que va-t-il se passer pour les utilisateurs du site web à la 6ème minute, et en quoi cela rend-il l'utilisation de **Vault Agent** (Niveau 5) absolument indispensable ?_


> **LE PROBLÉME**
>
> Si tu demandes à un développeur de gérer cela dans son code métier, il doit :
>
> - Implémenter un thread en arrière-plan qui surveille l'horloge.
> - Appeler l'API de Vault avant l'expiration.
> - Écrire les nouveaux fichiers .pem sur le disque dur du serveur.
> - Trouver un moyen de dire au serveur web (ex: Nginx, Uvicorn, Node.js) de "recharger" ses certificats à chaud sans couper les connexions en cours (graceful reload).
>
> C'est extrêmement complexe, source de bugs de concurrence, et cela mélange la logique métier de l'application avec de la plomberie cryptographique.
>
> **LA SOLUTION**
>
> C'est précisément pour cette raison que l'on déploie **Vault Agent**. L'Agent devient un processus séparé (un sidecar) qui tourne à côté de ton application dans le même conteneur ou serveur.
>
> Il gère silencieusement le renouvellement des baux, réécrit proprement le fichier de configuration ou les certificats sur le disque, et peut même exécuter une commande pour redémarrer le service à ta place, pendant que ton code métier reste totalement "aveugle" à cette complexité.

## Niveau 4 : Expert - Automatisation Déclarative via Vault Agent

Lorsque les développeurs opèrent des environnements où les applications ne peuvent pas intégrer le SDK HVAC, Vault Agent prend le relais pour résoudre l'intégration matérielle. Le démon Vault Agent exécute de manière autonome l'authentification (Auto-Auth) et maintient à jour les fichiers de configuration contenant les secrets via son moteur de Templating.

Jusqu'à présent, ton script Python utilisait la bibliothèque hvac pour parler à Vault. Il contenait toute la logique de sécurité.

Avec Vault Agent, nous supprimons totalement hvac du code de l'application. Ton code applicatif va redevenir "bête" et se contentera de lire un simple fichier texte local (ex: config_generee.txt). C'est un processus externe (l'Agent) qui va s'authentifier, récupérer les secrets, forger le certificat PKI, et écrire ce fichier sur le disque.

### 1. Les droits d'accès (Authorization / ACL) et Politique (Policy)

Puisque l'Agent est un processus autonome, il a besoin de fichiers de configuration sur le disque. Cette cellule va générer :

- Les identifiants `AppRole` dans des fichiers textes pour que l'Agent puisse s'authentifier (Auto-Auth).
- Un fichier modèle (`.tpl`) qui dicte à l'Agent la forme du fichier final qu'il doit générer.
- Le fichier de configuration HCL (`agent.hcl`) de l'Agent lui-même.

Authentification ≠ Autorisation.

L'Agent a bien montré son badge à l'entrée de Vault (il est bien le backend-microservice). Mais quand il va essayé d'ouvrir la porte de la salle des bases de données (`secret/data/database/config`) et la porte de la salle des certificats (`pki/issue/role-serveur-web`), le gardien de Vault (le moteur ACL) lui répondra : _"Tu es bien identifié, mais tu n'as pas l'habilitation de sécurité pour entrer ici."_

En effet, nous avons créé un rôle, des moteurs, mais nous n'avons jamais attaché de **politique (Policy)** autorisant ce rôle à lire ces chemins précis ! Par défaut, Vault applique un "Deny All" (refus total) absolu.


In [ ]:
# Cellule 14 : Le Grand Nettoyage (Reset de l'environnement)
import os

print("--- DESTRUCTION DE L'ANCIENNE CONFIGURATION ---")

# 1. Suppression du rôle corrompu
try:
    client.auth.approle.delete_role(role_name="backend-microservice")
    print("🧹 Rôle 'backend-microservice' supprimé de Vault.")
except Exception as e:
    print(f"Le rôle n'existait pas ou est déjà supprimé. ({e})")

# 2. Suppression de la politique
try:
    client.sys.delete_policy(name="agent-policy")
    print("🧹 Politique 'agent-policy' supprimée de Vault.")
except Exception as e:
    print(f"La politique n'existait pas. ({e})")

# 3. Suppression des fichiers locaux obsolètes
fichiers_a_supprimer = [
    "role_id.txt",
    "secret_id.txt",
    "app_config.tpl",
    "agent.hcl",
    "config_generee.txt",
]
for fichier in fichiers_a_supprimer:
    if os.path.exists(fichier):
        os.remove(fichier)
        print(f"🗑️ Fichier local '{fichier}' supprimé.")

print("✅ Environnement totalement nettoyé. Prêt pour une configuration propre.")

--- DESTRUCTION DE L'ANCIENNE CONFIGURATION ---
🧹 Rôle 'backend-microservice' supprimé de Vault.
🧹 Politique 'agent-policy' supprimée de Vault.
🗑️ Fichier local 'role_id.txt' supprimé.
🗑️ Fichier local 'secret_id.txt' supprimé.
🗑️ Fichier local 'app_config.tpl' supprimé.
🗑️ Fichier local 'agent.hcl' supprimé.
✅ Environnement totalement nettoyé. Prêt pour une configuration propre.


In [34]:
# Cellule 15 : La Configuration Propre et Déclarative (Vault Agent)
import os

print("--- CONSTRUCTION DE L'ARCHITECTURE ---")

# 1. Création de la politique de sécurité (Autorisation)
policy_hcl = """
path "secret/data/database/config" {
  capabilities = ["read"]
}
path "pki/issue/role-serveur-web" {
  capabilities = ["create", "update"]
}
"""
client.sys.create_or_update_policy(name="agent-policy", policy=policy_hcl)
print("🔒 1. Politique 'agent-policy' créée.")

# 2. Création de l'AppRole vierge (sans surcharger les TTL, on laisse Vault gérer)
client.auth.approle.create_or_update_approle(
    role_name="backend-microservice", token_policies=["agent-policy"]
)
print("🤖 2. AppRole 'backend-microservice' créé et lié à la politique.")

# 3. Génération et sauvegarde des identifiants (Authentification)
role_id_resp = client.auth.approle.read_role_id(role_name="backend-microservice")
role_id = role_id_resp["data"]["role_id"]

secret_id_resp = client.auth.approle.generate_secret_id(
    role_name="backend-microservice"
)
agent_secret_id = secret_id_resp["data"]["secret_id"]

with open("role_id.txt", "w") as f:
    f.write(role_id)
with open("secret_id.txt", "w") as f:
    f.write(agent_secret_id)
print("🔑 3. Nouveaux fichiers RoleID et SecretID générés sur le disque.")

# 4. Création du modèle (Template) de configuration
template_content = """
=== CONFIGURATION INJECTEE PAR VAULT AGENT ===

[DATABASE]
# L'Agent va chercher le secret statique du Niveau 1
{{ with secret "secret/data/database/config" }}
Mot de passe = {{ .Data.data.password }}
{{ end }}

[TLS]
# L'Agent génère un certificat dynamique du Niveau 4 (valide 5 minutes)
{{ with secret "pki/issue/role-serveur-web" "common_name=api.mon-app.local" "ttl=5m" }}
Certificat = {{ .Data.certificate }}
{{ end }}
"""
with open("app_config.tpl", "w") as f:
    f.write(template_content)
print("📄 4. Modèle 'app_config.tpl' créé.")

# 5. Création du fichier de configuration de l'Agent
hcl_content = """
vault {
  address = "http://127.0.0.1:8200"
}

auto_auth {
  method "approle" {
    config = {
      role_id_file_path = "role_id.txt"
      secret_id_file_path = "secret_id.txt"
      remove_secret_id_file_after_reading = false
    }
  }
}

template {
  source      = "app_config.tpl"
  destination = "config_generee.txt"
}
"""
with open("agent.hcl", "w") as f:
    f.write(hcl_content)
print("⚙️  5. Configuration 'agent.hcl' créée.")

print("\n✅ TOUT EST PRÊT !")

--- CONSTRUCTION DE L'ARCHITECTURE ---
🔒 1. Politique 'agent-policy' créée.
🤖 2. AppRole 'backend-microservice' créé et lié à la politique.
🔑 3. Nouveaux fichiers RoleID et SecretID générés sur le disque.
📄 4. Modèle 'app_config.tpl' créé.
⚙️  5. Configuration 'agent.hcl' créée.

✅ TOUT EST PRÊT !


**LE TEST FINAL : EXÉCUTER L'AGENT**

L'Agent est un démon (daemon). Il est fait pour tourner en arrière-plan en continu. Ne l'exécute pas dans une cellule Jupyter, sinon elle va tourner à l'infini.

La génération d'une configuration fonctionnelle pour le développement peut s'effectuer directement via le terminal avec la commande vault `agent generate-config`. Dans des environnements de production, le fichier `HCL agent-config.hcl` régit le comportement du démon :

- Ouvre ton terminal système (celui où tu as lancé ton serveur Vault ou un nouveau terminal à côté).
- Place-toi dans le même dossier que ton Notebook (là où les fichiers .txt, .tpl et .hcl viennent d'être créés).
- Lance cette commande **dans ton Terminal**:

```Bash
vault agent -config=agent.hcl
```


> #### 💡 Note d'Architecture : Le problème de l'œuf et de la poule (Bootstrapping)
>
> L'Agent Vault tourne actuellement en boucle dans votre terminal avec une erreur `no secret exists at secret/data/database/config`.
>
> **Pourquoi ?** Parce que notre Agent est configuré pour exiger deux choses avant de générer son fichier : un certificat dynamique (qu'il a réussi à créer) ET un mot de passe statique (qui n'existe pas dans le coffre). L'Agent préfère bloquer la production plutôt que de livrer un fichier partiel qui ferait planter l'application.
>
> **Où cela devrait-il se trouver idéalement ?**
> Dans une véritable infrastructure (ou au début de ce cours, au Niveau 1), ce secret statique est injecté lors de la phase d'amorçage (Bootstrapping) par l'équipe sécurité ou un outil comme Terraform, avant même que l'Agent ne démarre.
>
> Puisque notre serveur de développement (qui tourne en RAM) est vierge, nous allons agir en tant qu'Administrateur Sécurité et injecter ce secret manquant à chaud pour observer l'Agent se débloquer instantanément.


In [35]:
# Nous nous assurons que le moteur KV est actif (au cas où il aurait été désactivé)
try:
    client.sys.enable_secrets_engine(backend_type="kv-v2", path="secret")
except Exception:
    pass  # Le moteur est déjà actif

# Injection du secret attendu par le template de l'Agent
client.secrets.kv.v2.create_or_update_secret(
    path="database/config", secret=dict(password="SuperSecretDB2026!")
)
print("✅ Le secret statique 'database/config' a été injecté dans Vault.")
print(
    "👉 Regardez votre terminal : l'Agent devrait instantanément détecter la donnée et générer le fichier !"
)

✅ Le secret statique 'database/config' a été injecté dans Vault.
👉 Regardez votre terminal : l'Agent devrait instantanément détecter la donnée et générer le fichier !
